In [1]:
import json
import pandas as pd
import numpy as np
import os
os.chdir('..')

In [23]:
from src import load_finetuned_model, load_finetuned_model_lens_from_dir
from src.utils import postprocess_absa_outputs, calculate_metrics
import torch
import json
from tqdm import tqdm
import re
from glob import glob

In [24]:
dataset_folder = 'corrected_splitopinion_typocorrected_aos_arrow'
lang = 'indo'
test_json_path = f'hotel_dataset/{lang}/{dataset_folder}/hotel_aste_test_augmented.json'

In [25]:
with open(test_json_path, 'r') as f:
	test_data = json.load(f)

# Get the input prompts, labels, sentence IDs, and task elements
prompts = [instance['input'] for instance in test_data]
labels = [instance['target'] for instance in test_data]
sentence_ids = [instance['sentence_id'] for instance in test_data]
tasks = [instance['task_elements'] for instance in test_data]
element_orders = [instance['element_order'] for instance in test_data]

In [26]:
output_paths = glob(f'outputs/evals/eap/{dataset_folder}/circuit-{lang}_finetune-{lang}/**/*.json', recursive=True)
output_paths

['outputs/evals/eap/corrected_splitopinion_typocorrected_aos_arrow/circuit-indo_finetune-indo/seed_31415/aos_sequence_variants/full_sft/2025-10-10 21:16:22.986208_tflens_hotel_aste_train_augmented_noreasoning_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20/2025-10-10 21:16:22.986208_tflens_hotel_aste_train_augmented_noreasoning_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20/raw_inference_results.json',
 'outputs/evals/eap/corrected_splitopinion_typocorrected_aos_arrow/circuit-indo_finetune-indo/seed_777/aos_sequence_variants/full_sft/2025-10-10 21:16:23.522235_tflens_hotel_aste_train_augmented_noreasoning_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20/2025-10-10 21:16:23.522235_tflens_hotel_aste_train_augmented_noreasoning_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20/raw_inference_results.json',
 'outputs/evals/eap/corrected_splitopinion_typocorrected_aos_arrow/circuit-indo_finetune-indo/seed_9584/aos_sequence_variants/full_sft/2025-10-10 20:13:06.234684_tflens_hotel_aste_train_augmented_nore

In [27]:
for output_path in output_paths:
	with open(output_path, 'r') as f:
		outputs = json.load(f)
	inference_results = []

	per_task = {}
	for prompt, pred, label, si, t, element_order in zip(prompts, outputs, labels, sentence_ids, tasks, element_orders):

		# Split the target and prediction into lists
		target_split = label.split(" [SSEP] ")
		pred_split = pred.split(" [SSEP] ")
		# Strip whitespace
		target_split = [l.strip() for l in target_split]
		pred_split = [l.strip() for l in pred_split]
		
		# # Split the target and prediction into lists (GAS and LegoABSA)
		# target_split = label.split(';')
		# target_split = [l.strip() for l in target_split]
		# pred_split = pred.split(';')
		# pred_split = [l.strip() for l in pred_split]

		# Store inference results
		inf_dict = {}
		inf_dict["sentence_id"] = si
		inf_dict["task_elements"] = t
		inf_dict["element_order"] = element_order
		inf_dict["input"] = prompt
		inf_dict["target"] = label
		inf_dict["prediction"] = pred
		inf_dict["target_list"] = target_split
		inf_dict["prediction_list"] = pred_split

		inference_results.append(inf_dict)

		# Store predictions and targets per task for metric calculation
		if element_order not in per_task.keys():
			per_task[element_order] = {"predictions": [], "targets":[]}
		per_task[element_order]["predictions"].append(pred_split)
		per_task[element_order]["targets"].append(target_split)


	result_metrics = {}
	for task, v in per_task.items():
		predictions = v["predictions"]
		targets = v["targets"]
		result_metrics.update(
			calculate_metrics(predictions, targets, task)
	)
	scaled_result_metrics = {key: value * 100 for key, value in result_metrics.items()}

	print("Evaluation results:", scaled_result_metrics)
	
	# Save the metric results
	output_dir = os.path.dirname(output_path)
	output_file = os.path.join(output_dir, 'evaluation_results.json')
	with open(output_file, 'w') as f:
		json.dump(scaled_result_metrics, f, indent=4)
	print(f"Evaluation results saved to {output_file}")

	# Save the inference results
	inference_output_file = os.path.join(output_dir, 'inference_results.json')
	with open(inference_output_file, 'w') as f:
		json.dump(inference_results, f, indent=4)
	print(f"Inference results saved to {inference_output_file}")

Evaluation results: {'precision_aos': 60.77774052966812, 'recall_aos': 60.594919786096256, 'f1_aos': 60.68619246861925}
Evaluation results saved to outputs/evals/eap/corrected_splitopinion_typocorrected_aos_arrow/circuit-indo_finetune-indo/seed_31415/aos_sequence_variants/full_sft/2025-10-10 21:16:22.986208_tflens_hotel_aste_train_augmented_noreasoning_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20/2025-10-10 21:16:22.986208_tflens_hotel_aste_train_augmented_noreasoning_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20/evaluation_results.json
Inference results saved to outputs/evals/eap/corrected_splitopinion_typocorrected_aos_arrow/circuit-indo_finetune-indo/seed_31415/aos_sequence_variants/full_sft/2025-10-10 21:16:22.986208_tflens_hotel_aste_train_augmented_noreasoning_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20/2025-10-10 21:16:22.986208_tflens_hotel_aste_train_augmented_noreasoning_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20/inference_results.json
Evaluation results: {'precision_aos': 